# Dual-Branch Error Analysis

Analysis-only notebook for when and why dual-branch temporal dialogue memory helps or hurts. It does not train models and does not modify checkpoints or original prediction files.

## 1. Setup

In [1]:

from pathlib import Path
import os, json, warnings, re
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib_cache'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, mutual_info_score
try:
    import seaborn as sns
except Exception:
    sns = None
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'results').exists() and (PROJECT_ROOT.parent / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / 'results'
DUAL_DIR = RESULTS_DIR / 'dual_branch'
OUTPUT_DIR = DUAL_DIR / 'analysis'
PLOT_DIR = OUTPUT_DIR / 'plots'
CASE_DIR = OUTPUT_DIR / 'case_studies'
TIMELINE_DIR = OUTPUT_DIR / 'timelines'
for p in [OUTPUT_DIR, PLOT_DIR, CASE_DIR, TIMELINE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
PATHS = {
    'mal': RESULTS_DIR / 'wavlm_mal_no_tim' / 'predictions.csv',
    'tim': RESULTS_DIR / 'wavlm_tim' / 'predictions.csv',
    'dual': DUAL_DIR / 'predictions.csv',
    'dual_temporal_subset': DUAL_DIR / 'temporal_subset_metrics.json',
    'dual_diagnostics': DUAL_DIR / 'branch_gate_stats.json',
}
LABELS = ['angry', 'happy', 'neutral', 'sad']
TEMPORAL_COLUMNS = ['duration','gap_prev','overlap_prev','overlap_ratio','is_overlap','is_interrupting_prev','speaker_switch','short_response','long_pause']
RESIDUAL_COLUMNS = ['alpha_value','beta_value','dialogue_residual_norm','temporal_residual_norm']
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


PROJECT_ROOT = /Users/ngocbao/Documents/Document/research/main/speech/exps/demo
OUTPUT_DIR = /Users/ngocbao/Documents/Document/research/main/speech/exps/demo/results/dual_branch/analysis


## 2. Load Predictions

## 3. Merge Model Predictions

In [2]:

def standardize_prediction_frame(df, model_name):
    df = df.copy()
    rename = {'label':'gold_label','true_label':'gold_label','target_label':'gold_label','prediction':'pred_label','pred':'pred_label','speaker':'speaker_id'}
    df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
    missing = [c for c in ['dialogue_id','utterance_id','gold_label','pred_label'] if c not in df.columns]
    if missing:
        raise ValueError(f'{model_name} missing required columns: {missing}')
    for c in ['speaker_id','start_time','end_time']:
        if c not in df.columns: df[c] = np.nan
    for c in ['start_time','end_time'] + TEMPORAL_COLUMNS + RESIDUAL_COLUMNS:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in ['dialogue_id','utterance_id','speaker_id','gold_label','pred_label']:
        if c in df.columns: df[c] = df[c].astype(str)
    dup = df.duplicated(['dialogue_id','utterance_id']).sum()
    if dup:
        warnings.warn(f'{model_name}: dropping {dup} duplicate utterance rows')
        df = df.drop_duplicates(['dialogue_id','utterance_id'], keep='first')
    return df

def load_predictions(paths):
    frames = {}
    for name in ['mal','tim','dual']:
        path = paths[name]
        if not path.exists():
            warnings.warn(f'Missing {name} predictions: {path}')
            continue
        frames[name] = standardize_prediction_frame(pd.read_csv(path), name)
        print(f'Loaded {name}: {len(frames[name]):,} rows from {path.relative_to(PROJECT_ROOT)}')
    return frames

def load_iemocap_transcripts(root):
    rows=[]; base=root/'iemocap'
    if not base.exists():
        warnings.warn(f'IEMOCAP root not found: {base}')
        return pd.DataFrame(columns=['dialogue_id','utterance_id','transcript_text','audio_path'])
    pat=re.compile(r'^(?P<utt>\S+)\s+\[(?P<start>\d+(?:\.\d+)?)-(?P<end>\d+(?:\.\d+)?)\]:\s*(?P<text>.*)$')
    for path in sorted(base.glob('Session*/dialog/transcriptions/*.txt')):
        for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
            m=pat.match(line.strip())
            if not m: continue
            utt=m.group('utt'); did='_'.join(utt.split('_')[:-1]); session_num=str(int(utt[:5].replace('Ses','')))
            audio=base/f'Session{session_num}'/'sentences'/'wav'/did/f'{utt}.wav'
            rows.append({'dialogue_id':did,'utterance_id':utt,'transcript_text':m.group('text').strip(),'audio_path':str(audio.relative_to(root)) if audio.exists() else ''})
    return pd.DataFrame(rows).drop_duplicates(['dialogue_id','utterance_id'])

def recompute_temporal_features(df):
    df = df.copy()
    for c in TEMPORAL_COLUMNS:
        if c not in df.columns: df[c]=np.nan
    if not df[TEMPORAL_COLUMNS].isna().any(axis=None): return df
    out=[]
    for _,g in df.sort_values(['dialogue_id','start_time','end_time','utterance_id']).groupby('dialogue_id', sort=False):
        prev_end=None; prev_speaker=None
        for row in g.to_dict('records'):
            start=float(row.get('start_time', np.nan)); end=float(row.get('end_time', np.nan)); sp=str(row.get('speaker_id',''))
            dur=max(0.0,end-start) if np.isfinite(start) and np.isfinite(end) else np.nan
            has_prev=prev_end is not None and np.isfinite(start) and np.isfinite(prev_end)
            gap=start-prev_end if has_prev else 0.0; overlap=max(0.0,prev_end-start) if has_prev else 0.0; switch=1.0 if has_prev and sp != prev_speaker else 0.0
            row.update({'duration':dur,'gap_prev':gap,'overlap_prev':overlap,'overlap_ratio':overlap/max(dur,1e-6) if np.isfinite(dur) else 0.0,'is_overlap':1.0 if overlap>0.05 else 0.0,'is_interrupting_prev':1.0 if has_prev and switch and start<prev_end else 0.0,'speaker_switch':switch,'short_response':1.0 if has_prev and 0<=gap<0.3 else 0.0,'long_pause':1.0 if has_prev and gap>1.0 else 0.0})
            out.append(row); prev_end=end; prev_speaker=sp
    return pd.DataFrame(out)

def prefix_frame(df, model, include_features=False):
    prob=[c for c in df.columns if c.startswith('prob_')]
    cols=['dialogue_id','utterance_id','speaker_id','start_time','end_time','gold_label','pred_label']+prob
    if include_features: cols += [c for c in TEMPORAL_COLUMNS+RESIDUAL_COLUMNS if c in df.columns]
    out=df[[c for c in cols if c in df.columns]].copy()
    rename={'gold_label':f'{model}_gold_label','pred_label':f'{model}_pred_label'}
    rename.update({c:f'{model}_{c}' for c in prob})
    return out.rename(columns=rename)

frames = load_predictions(PATHS)
assert 'mal' in frames and 'dual' in frames, 'MAL and Dual predictions are required.'
frames['dual'] = recompute_temporal_features(frames['dual'])
transcripts = load_iemocap_transcripts(PROJECT_ROOT)
print('Transcript rows:', len(transcripts))

merged = prefix_frame(frames['dual'], 'dual', True)
for model in ['mal','tim']:
    if model in frames:
        merged = merged.merge(prefix_frame(frames[model], model), on=['dialogue_id','utterance_id'], how='outer', validate='one_to_one')
for col in ['speaker_id','start_time','end_time']:
    candidates=[c for c in [col,f'mal_{col}',f'tim_{col}'] if c in merged.columns]
    merged[col]=merged[candidates].bfill(axis=1).iloc[:,0]
    merged=merged.drop(columns=[c for c in candidates if c != col], errors='ignore')
gold_cols=[c for c in ['dual_gold_label','mal_gold_label','tim_gold_label'] if c in merged.columns]
if (merged[gold_cols].nunique(axis=1, dropna=True)>1).any(): warnings.warn('Gold-label mismatches found; keeping first non-null.')
merged['gold_label']=merged[gold_cols].bfill(axis=1).iloc[:,0]
merged['mal_correct']=merged['mal_pred_label'].eq(merged['gold_label'])
merged['tim_correct']=merged['tim_pred_label'].eq(merged['gold_label']) if 'tim_pred_label' in merged else np.nan
merged['dual_correct']=merged['dual_pred_label'].eq(merged['gold_label'])
merged['dual_improves_over_mal']=(~merged['mal_correct']) & merged['dual_correct']
merged['dual_hurts_vs_mal']=merged['mal_correct'] & (~merged['dual_correct'])
merged['both_correct']=merged['mal_correct'] & merged['dual_correct']
merged['both_wrong']=(~merged['mal_correct']) & (~merged['dual_correct'])
merged['error_category']=np.select([merged['dual_improves_over_mal'],merged['dual_hurts_vs_mal'],merged['both_correct']],['mal_wrong_dual_correct','mal_correct_dual_wrong','both_correct'],default='both_wrong')
merged=merged.merge(transcripts,on=['dialogue_id','utterance_id'],how='left')
print(merged[['dialogue_id','utterance_id','gold_label','mal_pred_label','dual_pred_label']].isna().sum())
print('Duplicate merged ids:', merged.duplicated(['dialogue_id','utterance_id']).sum())
merged.to_csv(OUTPUT_DIR/'merged_predictions.csv', index=False)
merged.head()


Loaded mal: 1,650 rows from results/wavlm_mal_no_tim/predictions.csv
Loaded tim: 1,650 rows from results/wavlm_tim/predictions.csv
Loaded dual: 1,650 rows from results/dual_branch/predictions.csv
Transcript rows: 10085
dialogue_id        0
utterance_id       0
gold_label         0
mal_pred_label     0
dual_pred_label    0
dtype: int64
Duplicate merged ids: 0


,dialogue_id,utterance_id,speaker_id_x,start_time_x,end_time_x,dual_gold_label,dual_pred_label,dual_prob_angry,dual_prob_happy,dual_prob_neutral,dual_prob_sad,duration,gap_prev,overlap_prev,overlap_ratio,is_overlap,is_interrupting_prev,speaker_switch,short_response,long_pause,alpha_value,beta_value,dialogue_residual_norm,temporal_residual_norm,speaker_id_y,start_time_y,end_time_y,mal_gold_label,mal_pred_label,mal_prob_angry,mal_prob_happy,mal_prob_neutral,mal_prob_sad,speaker_id,start_time,end_time,tim_gold_label,tim_pred_label,tim_prob_angry,tim_prob_happy,tim_prob_neutral,tim_prob_sad,gold_label,mal_correct,tim_correct,dual_correct,dual_improves_over_mal,dual_hurts_vs_mal,both_correct,both_wrong,error_category,transcript_text,audio_path
0,Ses05F_impro01,Ses05F_impro01_F000,Ses05_F,3.6132,6.17,neutral,neutral,0.169538,0.073564,0.659405,0.097493,2.5568,0.00,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.066755,0.041741,26.503094,12.809605,Ses05_F,3.6132,6.17,neutral,neutral,0.318639,0.115670,0.492520,0.073171,Ses05_F,3.6132,6.17,neutral,neutral,0.150415,0.053087,0.489137,0.307361,neutral,True,True,True,False,False,True,False,both_correct,"Hi, I need an ID.",iemocap/Session5/sentences/wav/Ses05F_impro01/...
1,Ses05F_impro01,Ses05F_impro01_F001,Ses05_F,14.1500,19.49,angry,neutral,0.169603,0.057754,0.680334,0.092309,5.3400,-0.63,0.63,0.117978,1.0,1.0,1.0,0.0,0.0,0.066755,0.041741,26.235744,14.728358,Ses05_F,14.1500,19.49,angry,neutral,0.244417,0.053570,0.651725,0.050288,Ses05_F,14.1500,19.49,angry,neutral,0.271964,0.010685,0.703208,0.014143,angry,False,False,False,False,False,False,True,both_wrong,"Okay, I'm sorry, but I just stood in this line...",iemocap/Session5/sentences/wav/Ses05F_impro01/...
2,Ses05F_impro01,Ses05F_impro01_F002,Ses05_F,22.8500,26.90,angry,neutral,0.286559,0.070125,0.541986,0.101331,4.0500,3.36,0.00,0.000000,0.0,0.0,0.0,0.0,1.0,0.066755,0.041741,18.840000,13.013433,Ses05_F,22.8500,26.90,angry,angry,0.559469,0.057966,0.309455,0.073110,Ses05_F,22.8500,26.90,angry,angry,0.772195,0.011687,0.173959,0.042159,angry,True,True,False,False,True,False,False,mal_correct_dual_wrong,"No, they told me-I'm sorry, but they told me t...",iemocap/Session5/sentences/wav/Ses05F_impro01/...
3,Ses05F_impro01,Ses05F_impro01_F003,Ses05_F,29.9800,34.46,angry,angry,0.761104,0.041193,0.184251,0.013452,4.4800,3.08,0.00,0.000000,0.0,0.0,0.0,0.0,1.0,0.066755,0.041741,41.317879,13.921580,Ses05_F,29.9800,34.46,angry,angry,0.912239,0.015502,0.066509,0.005750,Ses05_F,29.9800,34.46,angry,angry,0.944763,0.002822,0.051940,0.000475,angry,True,True,True,False,False,True,False,both_correct,"What, I mean what... what's the difference? W...",iemocap/Session5/sentences/wav/Ses05F_impro01/...
4,Ses05F_impro01,Ses05F_impro01_F004,Ses05_F,39.4800,42.18,angry,angry,0.693421,0.004823,0.300577,0.001180,2.7000,-1.40,1.40,0.518519,1.0,1.0,1.0,0.0,0.0,0.066755,0.041741,71.923164,29.583256,Ses05_F,39.4800,42.18,angry,angry,0.852419,0.002432,0.143637,0.001512,Ses05_F,39.4800,42.18,angry,angry,0.965986,0.000081,0.033885,0.000048,angry,True,True,True,False,False,True,False,both_correct,Can you just-can I just get the right-,iemocap/Session5/sentences/wav/Ses05F_impro01/...


## 4. Overall Metrics

## 5. Error Taxonomy

## 6. Temporal Interaction Score

In [3]:

def model_metrics(df, pred_col, model_name):
    sub=df[['gold_label',pred_col]].dropna(); y=sub['gold_label'].astype(str); p=sub[pred_col].astype(str)
    pr,rc,f1,s=precision_recall_fscore_support(y,p,labels=LABELS,zero_division=0)
    row={'model':model_name,'n':len(sub),'WA':accuracy_score(y,p),'UA':balanced_accuracy_score(y,p),'Macro-F1':float(np.mean(f1)),'WF1':precision_recall_fscore_support(y,p,average='weighted',zero_division=0)[2]}
    for label,a,b,c,d in zip(LABELS,pr,rc,f1,s): row.update({f'{label}_precision':a,f'{label}_recall':b,f'{label}_f1':c,f'{label}_support':int(d)})
    return row

# Overall metrics
metric_rows=[]
for model,col in [('MAL','mal_pred_label'),('TIM','tim_pred_label'),('Dual','dual_pred_label')]:
    if col in merged and merged[col].notna().any(): metric_rows.append(model_metrics(merged,col,model))
overall_metrics=pd.DataFrame(metric_rows); overall_metrics.to_csv(OUTPUT_DIR/'overall_metrics.csv',index=False); display(overall_metrics[['model','n','WA','UA','Macro-F1','WF1']])
ax=overall_metrics.set_index('model')[['WA','UA','Macro-F1','WF1']].plot(kind='bar',figsize=(9,4),ylim=(0,1),rot=0); ax.set_title('Overall Metrics'); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.savefig(PLOT_DIR/'overall_metrics_bar.png',dpi=160); plt.show()

# Error taxonomy
taxonomy=merged.error_category.value_counts().rename_axis('category').reset_index(name='count'); taxonomy['percentage']=taxonomy['count']/len(merged); taxonomy.to_csv(OUTPUT_DIR/'error_taxonomy.csv',index=False)
by_emotion=merged.groupby(['gold_label','error_category']).size().reset_index(name='count'); by_emotion['emotion_total']=by_emotion.groupby('gold_label')['count'].transform('sum'); by_emotion['percentage_within_emotion']=by_emotion['count']/by_emotion['emotion_total']; by_emotion.to_csv(OUTPUT_DIR/'error_taxonomy_by_emotion.csv',index=False)
display(taxonomy); display(by_emotion.pivot_table(index='gold_label',columns='error_category',values='count',fill_value=0))
ax=by_emotion.pivot_table(index='gold_label',columns='error_category',values='count',fill_value=0).reindex(LABELS).plot(kind='bar',stacked=True,figsize=(10,4)); ax.set_title('Error Taxonomy by Emotion'); plt.tight_layout(); plt.savefig(PLOT_DIR/'error_taxonomy_by_emotion.png',dpi=160); plt.show()
transition=pd.crosstab(merged['mal_pred_label'],merged['dual_pred_label']); transition.to_csv(OUTPUT_DIR/'mal_to_dual_prediction_transition.csv')
plt.figure(figsize=(7,5)); sns.heatmap(transition,annot=True,fmt='d',cmap='Blues') if sns else plt.imshow(transition.values,cmap='Blues'); plt.title('MAL pred -> Dual pred'); plt.tight_layout(); plt.savefig(PLOT_DIR/'mal_to_dual_prediction_transition.png',dpi=160); plt.show()

# Interaction score
for c in TEMPORAL_COLUMNS:
    if c not in merged: merged[c]=0.0
    merged[c]=pd.to_numeric(merged[c],errors='coerce').fillna(0.0)
merged['interaction_score']=merged['is_overlap']+merged['is_interrupting_prev']+0.5*merged['speaker_switch']+0.75*merged['short_response']+0.75*merged['long_pause']+np.minimum(merged['overlap_ratio'].clip(lower=0),1.0)
try: merged['interaction_bin']=pd.qcut(merged['interaction_score'].rank(method='first'),3,labels=['low','medium','high'])
except ValueError: merged['interaction_bin']=pd.cut(merged['interaction_score'],3,labels=['low','medium','high'])
merged['fixed_high_temporal_interaction']=(merged[['is_overlap','is_interrupting_prev','short_response','long_pause']]>0.5).any(axis=1)
rows=[]
for b,g in merged.groupby('interaction_bin',observed=False):
    row={'interaction_bin':str(b),'n':len(g),'dual_improvement_rate':g.dual_improves_over_mal.mean(),'dual_hurt_rate':g.dual_hurts_vs_mal.mean(),'fixed_high_rate':g.fixed_high_temporal_interaction.mean()}
    for model,col in [('MAL','mal_pred_label'),('Dual','dual_pred_label')]:
        m=model_metrics(g,col,model)
        for metric in ['WA','UA','Macro-F1','WF1']: row[f'{model}_{metric}']=m[metric]
    rows.append(row)
interaction_analysis=pd.DataFrame(rows); interaction_analysis.to_csv(OUTPUT_DIR/'interaction_score_analysis.csv',index=False); display(interaction_analysis)
merged.to_csv(OUTPUT_DIR/'merged_predictions.csv',index=False)
plt.figure(figsize=(7,4)); plt.hist(merged.interaction_score,bins=30); plt.title('Interaction Score Distribution'); plt.tight_layout(); plt.savefig(PLOT_DIR/'interaction_score_histogram.png',dpi=160); plt.show()
ax=interaction_analysis.set_index('interaction_bin')[['dual_improvement_rate','dual_hurt_rate']].plot(kind='bar',figsize=(8,4),rot=0); ax.set_title('Improvement/Hurt by Interaction Bin'); plt.tight_layout(); plt.savefig(PLOT_DIR/'improvement_hurt_by_interaction_bin.png',dpi=160); plt.show()
ax=interaction_analysis.set_index('interaction_bin')[['MAL_Macro-F1','Dual_Macro-F1']].plot(kind='bar',figsize=(8,4),rot=0,ylim=(0,1)); ax.set_title('Macro-F1 by Interaction Bin'); plt.tight_layout(); plt.savefig(PLOT_DIR/'macro_f1_by_interaction_bin.png',dpi=160); plt.show()


,model,n,WA,UA,Macro-F1,WF1
0,MAL,1650,0.701212,0.684004,0.689376,0.700969
1,TIM,1650,0.713333,0.695906,0.700498,0.715007
2,Dual,1650,0.681818,0.671063,0.671275,0.684940


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=overall_metrics.set_index('model')[['WA','UA','Macro-F1','WF1']].plot(kind='bar',figsize=(9,4),ylim=(0,1),rot=0); ax.set_title('Overall Metrics'); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.savefig(PLOT_DIR/'overall_metrics_bar.png',dpi=160); plt.show()


,category,count,percentage
0,both_correct,1024,0.620606
1,both_wrong,392,0.237576
2,mal_correct_dual_wrong,133,0.080606
3,mal_wrong_dual_correct,101,0.061212


error_category,both_correct,both_wrong,mal_correct_dual_wrong,mal_wrong_dual_correct
gold_label,,,,
angry,402.0,90.0,41.0,18.0
happy,282.0,128.0,36.0,14.0
neutral,201.0,101.0,36.0,46.0
sad,139.0,73.0,20.0,23.0


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=by_emotion.pivot_table(index='gold_label',columns='error_category',values='count',fill_value=0).reindex(LABELS).plot(kind='bar',stacked=True,figsize=(10,4)); ax.set_title('Error Taxonomy by Emotion'); plt.tight_layout(); plt.savefig(PLOT_DIR/'error_taxonomy_by_emotion.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(7,5)); sns.heatmap(transition,annot=True,fmt='d',cmap='Blues') if sns else plt.imshow(transition.values,cmap='Blues'); plt.title('MAL pred -> Dual pred'); plt.tight_layout(); plt.savefig(PLOT_DIR/'mal_to_dual_prediction_transition.png',dpi=160); plt.show()


,interaction_bin,n,dual_improvement_rate,dual_hurt_rate,fixed_high_rate,MAL_WA,MAL_UA,MAL_Macro-F1,MAL_WF1,Dual_WA,Dual_UA,Dual_Macro-F1,Dual_WF1
0,low,550,0.069091,0.120000,0.738182,0.678182,0.675059,0.673758,0.681322,0.627273,0.629968,0.624093,0.629699
1,medium,550,0.050909,0.056364,1.000000,0.701818,0.671842,0.676426,0.699159,0.696364,0.665629,0.666547,0.696924
2,high,550,0.063636,0.065455,1.000000,0.723636,0.653790,0.667094,0.719198,0.721818,0.668499,0.680872,0.722802


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(7,4)); plt.hist(merged.interaction_score,bins=30); plt.title('Interaction Score Distribution'); plt.tight_layout(); plt.savefig(PLOT_DIR/'interaction_score_histogram.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=interaction_analysis.set_index('interaction_bin')[['dual_improvement_rate','dual_hurt_rate']].plot(kind='bar',figsize=(8,4),rot=0); ax.set_title('Improvement/Hurt by Interaction Bin'); plt.tight_layout(); plt.savefig(PLOT_DIR/'improvement_hurt_by_interaction_bin.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/91344081.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=interaction_analysis.set_

## 7. Temporal Feature Distribution by Outcome

## 8. Residual Contribution Analysis

## 9. Emotion-Wise Gain Analysis

## 10. Confusion Matrix Comparison

In [4]:

# Feature distributions
cont=[c for c in ['duration','gap_prev','overlap_prev','overlap_ratio','interaction_score','dialogue_residual_norm','temporal_residual_norm','beta_value'] if c in merged]
bincols=[c for c in ['is_overlap','is_interrupting_prev','speaker_switch','short_response','long_pause'] if c in merged]
summary=[]
for cat,g in merged.groupby('error_category'):
    for feat in cont:
        v=pd.to_numeric(g[feat],errors='coerce').dropna(); summary.append({'error_category':cat,'feature':feat,'type':'continuous','count':len(v),'mean':v.mean() if len(v) else np.nan,'std':v.std() if len(v)>1 else np.nan,'median':v.median() if len(v) else np.nan,'q25':v.quantile(.25) if len(v) else np.nan,'q75':v.quantile(.75) if len(v) else np.nan,'rate':np.nan})
    for feat in bincols:
        v=pd.to_numeric(g[feat],errors='coerce').dropna(); summary.append({'error_category':cat,'feature':feat,'type':'binary','count':len(v),'mean':np.nan,'std':np.nan,'median':np.nan,'q25':np.nan,'q75':np.nan,'rate':v.mean() if len(v) else np.nan})
feature_by_error=pd.DataFrame(summary); feature_by_error.to_csv(OUTPUT_DIR/'feature_by_error_category.csv',index=False); display(feature_by_error.head(20))
for feat in cont:
    plt.figure(figsize=(9,4)); data=[merged.loc[merged.error_category==cat,feat].dropna() for cat in sorted(merged.error_category.unique())]
    plt.boxplot(data,labels=sorted(merged.error_category.unique()),showfliers=False); plt.title(f'{feat} by Error Category'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(PLOT_DIR/f'boxplot_{feat}_by_error_category.png',dpi=160); plt.close()
rates=feature_by_error[feature_by_error.type=='binary'].pivot(index='error_category',columns='feature',values='rate')
if not rates.empty:
    ax=rates.plot(kind='bar',figsize=(10,4),rot=30,ylim=(0,1)); ax.set_title('Binary Feature Rates by Error Category'); plt.tight_layout(); plt.savefig(PLOT_DIR/'binary_feature_rates_by_error_category.png',dpi=160); plt.show()

# Residual contribution
if {'dialogue_residual_norm','temporal_residual_norm'}.issubset(merged.columns):
    merged['temporal_dialogue_residual_ratio']=merged.temporal_residual_norm/(merged.dialogue_residual_norm+1e-6)
    residual_analysis=merged.groupby(['interaction_bin','error_category','gold_label'],observed=False).agg(n=('utterance_id','count'),mean_dialogue_residual_norm=('dialogue_residual_norm','mean'),mean_temporal_residual_norm=('temporal_residual_norm','mean'),mean_residual_ratio=('temporal_dialogue_residual_ratio','mean'),mean_interaction_score=('interaction_score','mean')).reset_index()
    residual_analysis.to_csv(OUTPUT_DIR/'residual_analysis.csv',index=False); display(residual_analysis.head(20))
    ax=merged.boxplot(column='temporal_residual_norm',by='interaction_bin',figsize=(7,4),showfliers=False); plt.suptitle(''); ax.set_title('Temporal Residual by Interaction Bin'); plt.tight_layout(); plt.savefig(PLOT_DIR/'temporal_residual_by_interaction_bin.png',dpi=160); plt.show()
    ax=merged.boxplot(column='temporal_dialogue_residual_ratio',by='error_category',figsize=(9,4),showfliers=False); plt.suptitle(''); ax.set_title('Residual Ratio by Outcome'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(PLOT_DIR/'residual_ratio_by_outcome.png',dpi=160); plt.show()
    ax=merged.groupby('gold_label')[['dialogue_residual_norm','temporal_residual_norm']].mean().reindex(LABELS).plot(kind='bar',figsize=(8,4),rot=0); ax.set_title('Residual Norm by Emotion'); plt.tight_layout(); plt.savefig(PLOT_DIR/'residual_norm_by_emotion.png',dpi=160); plt.show()
    plt.figure(figsize=(6,4)); plt.scatter(merged.interaction_score,merged.temporal_residual_norm,s=12,alpha=.45); plt.xlabel('interaction_score'); plt.ylabel('temporal_residual_norm'); plt.tight_layout(); plt.savefig(PLOT_DIR/'interaction_vs_temporal_residual_scatter.png',dpi=160); plt.show()
else:
    residual_analysis=pd.DataFrame(); warnings.warn('Residual columns unavailable.')

# Emotion-wise gain
def class_table(pred_col,prefix):
    p,r,f1,s=precision_recall_fscore_support(merged.gold_label,merged[pred_col],labels=LABELS,zero_division=0)
    return pd.DataFrame({'gold_label':LABELS,f'{prefix}_precision':p,f'{prefix}_recall':r,f'{prefix}_f1':f1,'support':s})
emotion_gain=class_table('mal_pred_label','mal').merge(class_table('dual_pred_label','dual'),on=['gold_label','support'])
emotion_gain['recall_gain']=emotion_gain.dual_recall-emotion_gain.mal_recall; emotion_gain['f1_gain']=emotion_gain.dual_f1-emotion_gain.mal_f1
counts=merged.groupby('gold_label').agg(mal_wrong_dual_correct=('dual_improves_over_mal','sum'),mal_correct_dual_wrong=('dual_hurts_vs_mal','sum')).reset_index()
emotion_gain=emotion_gain.merge(counts,on='gold_label',how='left'); emotion_gain.to_csv(OUTPUT_DIR/'emotion_wise_gain.csv',index=False); display(emotion_gain)
for metric in ['recall_gain','f1_gain']:
    ax=emotion_gain.set_index('gold_label')[metric].reindex(LABELS).plot(kind='bar',figsize=(7,4),rot=0); ax.axhline(0,color='black'); ax.set_title(metric); plt.tight_layout(); plt.savefig(PLOT_DIR/f'{metric}_by_emotion.png',dpi=160); plt.show()

# Confusion matrices
def save_confusion(name,pred_col):
    cm=confusion_matrix(merged.gold_label,merged[pred_col],labels=LABELS); frame=pd.DataFrame(cm,index=LABELS,columns=LABELS); frame.to_csv(OUTPUT_DIR/f'confusion_{name}.csv')
    plt.figure(figsize=(6,5)); sns.heatmap(frame,annot=True,fmt='d',cmap='Blues') if sns else plt.imshow(frame.values,cmap='Blues'); plt.title(f'{name.upper()} Confusion'); plt.tight_layout(); plt.savefig(PLOT_DIR/f'confusion_{name}.png',dpi=160); plt.show(); return frame
conf_mal=save_confusion('mal','mal_pred_label'); conf_dual=save_confusion('dual','dual_pred_label')
if 'tim_pred_label' in merged and merged.tim_pred_label.notna().any(): conf_tim=save_confusion('tim','tim_pred_label')
delta=conf_dual-conf_mal; delta.to_csv(OUTPUT_DIR/'confusion_delta_dual_minus_mal.csv'); plt.figure(figsize=(6,5)); sns.heatmap(delta,annot=True,fmt='d',cmap='RdBu_r',center=0) if sns else plt.imshow(delta.values,cmap='RdBu_r'); plt.title('Delta Confusion: Dual - MAL'); plt.tight_layout(); plt.savefig(PLOT_DIR/'confusion_delta_dual_minus_mal.png',dpi=160); plt.show()


,error_category,feature,type,count,mean,std,median,q25,q75,rate
0,both_correct,duration,continuous,1024,4.514523,3.419705,3.475000,2.157500,5.900000,NaN
1,both_correct,gap_prev,continuous,1024,0.455523,3.478982,-0.390000,-0.802500,0.450000,NaN
2,both_correct,overlap_prev,continuous,1024,0.614280,0.943636,0.390000,0.000000,0.802500,NaN
3,both_correct,overlap_ratio,continuous,1024,0.244575,0.519364,0.085332,0.000000,0.265156,NaN
4,both_correct,interaction_score,continuous,1024,2.048404,1.033534,2.585332,0.750000,2.765156,NaN
5,both_correct,dialogue_residual_norm,continuous,1024,55.761175,19.377746,56.872240,40.267537,68.829443,NaN
6,both_correct,temporal_residual_norm,continuous,1024,36.717740,13.155167,38.554464,26.223775,47.148630,NaN
7,both_correct,beta_value,continuous,1024,0.041741,0.000000,0.041741,0.041741,0.041741,NaN
8,both_correct,is_overlap,binary,1024,NaN,NaN,NaN,NaN,NaN,0.627930
9,both_correct,is_interrupting_prev,binary,1024,NaN,NaN,NaN,NaN,NaN,0.634766


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data,labels=sorted(merged.error_category.unique()),showfliers=False); plt.title(f'{feat} by Error Category'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(PLOT_DIR/f'boxplot_{feat}_by_error_category.png',dpi=160); plt.close()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data,labels=sorted(merged.error_category.unique()),showfliers=False); plt.title(f'{feat} by Error Category'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(PLOT_DIR/f'boxplot_{feat}_by_error_category.p

,interaction_bin,error_category,gold_label,n,mean_dialogue_residual_norm,mean_temporal_residual_norm,mean_residual_ratio,mean_interaction_score
0,low,both_correct,angry,94,59.911761,40.093059,0.773298,0.662234
1,low,both_correct,happy,82,55.983151,32.140942,0.680960,0.710366
2,low,both_correct,neutral,50,37.018291,34.475176,1.010967,0.525000
3,low,both_correct,sad,81,50.900992,42.019753,0.864870,0.657407
4,low,both_wrong,angry,29,37.087194,36.842833,1.072712,0.663793
5,low,both_wrong,happy,54,45.233715,27.898650,0.676165,0.634259
6,low,both_wrong,neutral,28,45.044944,33.156992,0.796660,0.696429
7,low,both_wrong,sad,28,35.194949,24.343437,0.696449,0.491071
8,low,mal_correct_dual_wrong,angry,22,35.791763,38.325539,1.142983,0.727273
9,low,mal_correct_dual_wrong,happy,20,32.386784,25.724489,0.998506,0.525000


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=merged.boxplot(column='temporal_residual_norm',by='interaction_bin',figsize=(7,4),showfliers=False); plt.suptitle(''); ax.set_title('Temporal Residual by Interaction Bin'); plt.tight_layout(); plt.savefig(PLOT_DIR/'temporal_residual_by_interaction_bin.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=merged.boxplot(column='temporal_dialogue_residual_ratio',by='error_category',figsize=(9,4),showfliers=False); plt.suptitle(''); ax.set_title('Residual Ratio by Outcome'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(PLOT_DIR/'residual_ratio_by_outcome.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:25: UserWarning: FigureC

,gold_label,mal_precision,mal_recall,mal_f1,support,dual_precision,dual_recall,dual_f1,recall_gain,f1_gain,mal_wrong_dual_correct,mal_correct_dual_wrong
0,angry,0.691108,0.803993,0.743289,551,0.711864,0.762250,0.736196,-0.041742,-0.007092,18,41
1,happy,0.823834,0.691304,0.751773,460,0.857971,0.643478,0.735404,-0.047826,-0.016369,14,36
2,neutral,0.612403,0.617188,0.614786,384,0.553812,0.643229,0.595181,0.026042,-0.019605,46,36
3,sad,0.673729,0.623529,0.647658,255,0.602230,0.635294,0.618321,0.011765,-0.029337,23,20


/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=emotion_gain.set_index('gold_label')[metric].reindex(LABELS).plot(kind='bar',figsize=(7,4),rot=0); ax.axhline(0,color='black'); ax.set_title(metric); plt.tight_layout(); plt.savefig(PLOT_DIR/f'{metric}_by_emotion.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=emotion_gain.set_index('gold_label')[metric].reindex(LABELS).plot(kind='bar',figsize=(7,4),rot=0); ax.axhline(0,color='black'); ax.set_title(metric); plt.tight_layout(); plt.savefig(PLOT_DIR/f'{metric}_by_emotion.png',dpi=160); plt.show()
/var/folders/62/f6sjw4bd0td2_b5c63kfyh0c0000gn/T/ipykernel_6856/249211378.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(6,5)); sns.heatmap(frame,

## 11. Case Study Selection

## 12. Dialogue Timeline Visualization

## 13. Feature Importance Sanity Checks

## 14. Summary Report

In [5]:

# Case studies
case_cols=[c for c in ['dialogue_id','utterance_id','speaker_id','start_time','end_time','gold_label','mal_pred_label','dual_pred_label','tim_pred_label','duration','gap_prev','overlap_prev','overlap_ratio','is_overlap','is_interrupting_prev','speaker_switch','short_response','long_pause','interaction_score','dialogue_residual_norm','temporal_residual_norm','transcript_text','audio_path'] if c in merged]
def save_cases(name, frame, sort_cols):
    sort_cols=[c for c in sort_cols if c in frame]
    out=frame.sort_values(sort_cols,ascending=[False]*len(sort_cols)) if sort_cols else frame.copy(); out=out[case_cols]; out.to_csv(CASE_DIR/f'{name}.csv',index=False); return out
high=merged.interaction_bin.astype(str).eq('high'); low=merged.interaction_bin.astype(str).eq('low')
dual_improves_cases=save_cases('dual_improves_cases',merged[merged.dual_improves_over_mal & high],['temporal_residual_norm','interaction_score'])
dual_hurts_cases=save_cases('dual_hurts_cases',merged[merged.dual_hurts_vs_mal & high],['temporal_residual_norm','interaction_score'])
both_wrong_high_interaction_cases=save_cases('both_wrong_high_interaction_cases',merged[merged.both_wrong & high],['temporal_residual_norm','interaction_score'])
low_interaction_no_gain_cases=save_cases('low_interaction_no_gain_cases',merged[low & ((merged.mal_pred_label==merged.dual_pred_label)|merged.both_correct)],['interaction_score'])
print({'improves':len(dual_improves_cases),'hurts':len(dual_hurts_cases),'both_wrong_high':len(both_wrong_high_interaction_cases),'low_no_gain':len(low_interaction_no_gain_cases)})

# Timeline visualization
CATEGORY_COLORS={'mal_wrong_dual_correct':'#59a14f','mal_correct_dual_wrong':'#e15759','both_correct':'#4e79a7','both_wrong':'#9c755f'}
def plot_dialogue_timeline(dialogue_id, center_utterance_id=None, window=5, color_by='category', save_path=None):
    df=merged[merged.dialogue_id==dialogue_id].sort_values(['start_time','end_time','utterance_id']).copy()
    if center_utterance_id is not None and center_utterance_id in set(df.utterance_id):
        idx=int(np.flatnonzero(df.utterance_id.to_numpy()==center_utterance_id)[0]); df=df.iloc[max(0,idx-window):min(len(df),idx+window+1)].copy()
    speakers={s:i for i,s in enumerate(sorted(df.speaker_id.astype(str).unique()))}
    fig,ax=plt.subplots(figsize=(12,max(3,.45*len(df))))
    for _,row in df.iterrows():
        start=float(row.start_time); end=float(row.end_time); width=max(end-start,.05); y=speakers[str(row.speaker_id)]
        edge='black' if row.utterance_id==center_utterance_id else 'white'; lw=2.5 if row.utterance_id==center_utterance_id else .8
        ax.barh(y,width,left=start,height=.35,color=CATEGORY_COLORS.get(row.error_category,'#bab0ac'),edgecolor=edge,linewidth=lw,alpha=.9)
        ax.text(start+width/2,y+.22,f"{str(row.utterance_id).split('_')[-1]} | gold={row.gold_label} | M={row.mal_pred_label} | D={row.dual_pred_label}",ha='center',va='bottom',fontsize=8)
        if row.get('overlap_prev',0)>0: ax.axvspan(start,min(end,start+float(row.overlap_prev)),color='red',alpha=.12)
    ax.set_yticks(list(speakers.values())); ax.set_yticklabels(list(speakers.keys())); ax.set_xlabel('time (s)'); ax.set_title(f'Dialogue timeline: {dialogue_id}'); ax.grid(axis='x',alpha=.25); plt.tight_layout()
    if save_path: fig.savefig(save_path,dpi=160)
    return fig
for prefix,cases in [('improves',dual_improves_cases),('hurts',dual_hurts_cases)]:
    for i,row in cases.head(10).reset_index(drop=True).iterrows():
        fig=plot_dialogue_timeline(row.dialogue_id,row.utterance_id,window=5,save_path=TIMELINE_DIR/f"{prefix}_{i+1:02d}_{row.dialogue_id}_{row.utterance_id}.png")
        plt.close(fig)
print('Saved timelines:', TIMELINE_DIR)

# Feature association
assoc=[]; features=[c for c in TEMPORAL_COLUMNS+['interaction_score','dialogue_residual_norm','temporal_residual_norm'] if c in merged]
for feat in features:
    x=pd.to_numeric(merged[feat],errors='coerce').fillna(0.0)
    for target in ['dual_improves_over_mal','dual_hurts_vs_mal']:
        y=merged[target].astype(int); row={'feature':feat,'target':target}
        if x.nunique()>1 and y.nunique()>1:
            row['pearson_corr']=float(np.corrcoef(x,y)[0,1]); row['spearman_corr']=float(spearmanr(x,y).correlation) if spearmanr else np.nan
            try:
                bins=pd.qcut(x.rank(method='first'),q=min(5,max(2,x.nunique())),duplicates='drop'); row['mutual_information']=float(mutual_info_score(bins.astype(str),y.astype(str)))
            except Exception: row['mutual_information']=np.nan
        else: row.update({'pearson_corr':np.nan,'spearman_corr':np.nan,'mutual_information':np.nan})
        assoc.append(row)
feature_association=pd.DataFrame(assoc).sort_values(['target','mutual_information'],ascending=[True,False]); feature_association.to_csv(OUTPUT_DIR/'feature_association_with_improvement.csv',index=False); display(feature_association.head(30))

# Markdown report
def pct(x): return f'{100*x:.2f}%'
mal_row=overall_metrics[overall_metrics.model=='MAL'].iloc[0]; dual_row=overall_metrics[overall_metrics.model=='Dual'].iloc[0]
counts=merged.error_category.value_counts(); improve=int(counts.get('mal_wrong_dual_correct',0)); hurt=int(counts.get('mal_correct_dual_wrong',0))
best=emotion_gain.sort_values('f1_gain',ascending=False).iloc[0]; worst=emotion_gain.sort_values('f1_gain').iloc[0]
interaction_line='; '.join([f"{r.interaction_bin}: improve={pct(r.dual_improvement_rate)}, hurt={pct(r.dual_hurt_rate)}" for r in interaction_analysis.itertuples()])
residual_line='Residual norm columns unavailable.'
if 'temporal_residual_norm' in merged: residual_line='Mean temporal residual norm by interaction bin: '+', '.join(f'{k}={v:.3f}' for k,v in merged.groupby('interaction_bin',observed=False).temporal_residual_norm.mean().items())
report_lines=[
    '# Dual-Branch Error Analysis Report','',
    '## Overall Finding','',
    f"MAL UA = {mal_row['UA']:.4f}, Dual UA = {dual_row['UA']:.4f}, delta = {dual_row['UA']-mal_row['UA']:+.4f}.",
    f"MAL Macro-F1 = {mal_row['Macro-F1']:.4f}, Dual Macro-F1 = {dual_row['Macro-F1']:.4f}, delta = {dual_row['Macro-F1']-mal_row['Macro-F1']:+.4f}.",'',
    '## Does Dual Improve Over MAL','',
    f'- MAL wrong / Dual correct: {improve} utterances ({pct(improve/len(merged))})',
    f'- MAL correct / Dual wrong: {hurt} utterances ({pct(hurt/len(merged))})',
    f"- Both correct: {int(counts.get('both_correct',0))}",
    f"- Both wrong: {int(counts.get('both_wrong',0))}",'',
    '## Emotion Effects','',
    f"Best F1 gain: {best['gold_label']} ({best['f1_gain']:+.4f}).",
    f"Worst F1 gain: {worst['gold_label']} ({worst['f1_gain']:+.4f}).",'',
    '## Temporal Interaction Subsets','', interaction_line, '',
    '## Residual/Gate Contribution','', residual_line, '',
    '## Recommended Next Fixes','',
    '1. Reduce temporal branch influence on low-interaction utterances.',
    '2. Add adaptive beta per utterance instead of a single global beta.',
    '3. Add feature-group gates for duration, gap, overlap, speaker switch, and turn position.',
    '4. Inspect overlap-driven false positives and recalibrate noisy temporal cues.',
    '5. Keep transcript/audio metadata for analysis and demo only unless defining a new multimodal baseline.'
]
report='\n'.join(report_lines); (OUTPUT_DIR/'dual_branch_error_analysis_report.md').write_text(report,encoding='utf-8'); print(report)


{'improves': 35, 'hurts': 36, 'both_wrong_high': 117, 'low_no_gain': 431}
Saved timelines: /Users/ngocbao/Documents/Document/research/main/speech/exps/demo/results/dual_branch/analysis/timelines


,feature,target,pearson_corr,spearman_corr,mutual_information
21,dialogue_residual_norm,dual_hurts_vs_mal,-0.167828,-0.168145,0.016292
19,interaction_score,dual_hurts_vs_mal,-0.096349,-0.087597,0.005618
7,overlap_ratio,dual_hurts_vs_mal,0.003056,-0.080634,0.005555
5,overlap_prev,dual_hurts_vs_mal,-0.020429,-0.084621,0.005491
3,gap_prev,dual_hurts_vs_mal,0.056167,0.078477,0.005052
11,is_interrupting_prev,dual_hurts_vs_mal,-0.099705,-0.099705,0.003821
9,is_overlap,dual_hurts_vs_mal,-0.093141,-0.093141,0.003434
23,temporal_residual_norm,dual_hurts_vs_mal,0.020032,0.022868,0.002881
1,duration,dual_hurts_vs_mal,0.050207,0.035901,0.001498
15,short_response,dual_hurts_vs_mal,0.055727,0.055727,0.001317


# Dual-Branch Error Analysis Report

## Overall Finding

MAL UA = 0.6840, Dual UA = 0.6711, delta = -0.0129.
MAL Macro-F1 = 0.6894, Dual Macro-F1 = 0.6713, delta = -0.0181.

## Does Dual Improve Over MAL

- MAL wrong / Dual correct: 101 utterances (6.12%)
- MAL correct / Dual wrong: 133 utterances (8.06%)
- Both correct: 1024
- Both wrong: 392

## Emotion Effects

Best F1 gain: angry (-0.0071).
Worst F1 gain: sad (-0.0293).

## Temporal Interaction Subsets

low: improve=6.91%, hurt=12.00%; medium: improve=5.09%, hurt=5.64%; high: improve=6.36%, hurt=6.55%

## Residual/Gate Contribution

Mean temporal residual norm by interaction bin: low=35.118, medium=37.510, high=36.573

## Recommended Next Fixes

1. Reduce temporal branch influence on low-interaction utterances.
2. Add adaptive beta per utterance instead of a single global beta.
3. Add feature-group gates for duration, gap, overlap, speaker switch, and turn position.
4. Inspect overlap-driven false positives and recalibrate noisy te